# Full EIG-Based Clarification Pipeline

`Problem Identification -> Candidate Solution Generation -> Interactive
Clarification Loop -> Oracle Interaction -> Belief Update Mechanism ->
Ask-or-Submit Decision -> Final Implementation -> Performance Evaluation`

All logic lives in 4 files next to this notebook:
`generate.py`, `evaluator.py`, `scoring.py`, `clarification.py`.
This notebook just wires them together and runs the evaluation.


In [ ]:
import sys
sys.path.append(".")   

import os, json
from tqdm import tqdm
from datasets import load_dataset

from clarification import run_clarification_pipeline
from evaluator import evaluate_final
from generate import N_CANDIDATES, M_TESTS, K_MAX, EPS, GAMMA, ETA, LAMBDA, RHO

print(f"N_CANDIDATES={N_CANDIDATES}  M_TESTS={M_TESTS}  K_MAX={K_MAX}")


In [ ]:
dataset = load_dataset("google-research-datasets/mbpp")
dataset


## Quick single-problem test

In [ ]:
sample = dataset["train"][0]
print(sample["text"])

result = run_clarification_pipeline(sample, verbose=True)

print("\nQuestions asked:", result["questions_asked"])
print("Final weights:", [round(w, 3) for w in result["weights"]])
print("\nFinal chosen code:\n", result["final_code"])


## Full run -- compute pass@1

In [ ]:
N_PROBLEMS = 20
SPLIT = "train"

results = []
passed = 0

for i in tqdm(range(N_PROBLEMS)):
    sample = dataset[SPLIT][i]
    try:
        pipeline_result = run_clarification_pipeline(sample)
        success, error = evaluate_final(
            pipeline_result["final_code"], sample["test_list"],
            setup_code=sample.get("test_setup_code", "")
        )
    except Exception as e:
        success, error = False, f"pipeline error: {e}"
        pipeline_result = {"final_code": None, "questions_asked": 0}

    if success:
        passed += 1

    results.append({
        "idx": i, "task_id": sample.get("task_id", i), "passed": success,
        "error": error, "questions_asked": pipeline_result["questions_asked"],
        "final_code": pipeline_result["final_code"],
    })

pass_at_1 = passed / N_PROBLEMS * 100
print(f"\nEIG Pipeline -- Passed: {passed}/{N_PROBLEMS} ({pass_at_1:.1f}%)")

os.makedirs("results", exist_ok=True)
with open(f"results/eig_pipeline_{SPLIT}_{N_PROBLEMS}.json", "w") as f:
    json.dump({
        "n": N_PROBLEMS, "split": SPLIT, "passed": passed, "pass_at_1": pass_at_1,
        "hyperparameters": {"N_CANDIDATES": N_CANDIDATES, "M_TESTS": M_TESTS, "K_MAX": K_MAX,
                             "EPS": EPS, "GAMMA": GAMMA, "ETA": ETA, "LAMBDA": LAMBDA, "RHO": RHO},
        "results": results,
    }, f, indent=2)

print(f"Saved to results/eig_pipeline_{SPLIT}_{N_PROBLEMS}.json")
